In [1]:
# Standard library
import os
import random
import time
from pathlib import Path

# Data
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader
)

# Torchvision
import torchvision
import torchvision.transforms as transforms

# Progress bars
from tqdm import tqdm

# Experiment tracking
import wandb


plt.style.use("default")
import sys
from dataclasses import dataclass

In [2]:
# 1. Clone your clean repository code from GitHub
REPO_URL = "https://github.com/K0NSTANT1N3/Facial-Expression-Recognition.git"
REPO_NAME = "Facial-Expression-Recognition"

if not os.path.exists(REPO_NAME):
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL}")
else:
    print("Repository already exists. Pulling latest updates...")
    os.system(f"cd {REPO_NAME} && git pull")

# 2. THE SECRET SAUCE: Add your cloned repo directory to Python's path
sys.path.append(os.path.abspath(REPO_NAME))

print("Successfully linked GitHub repository and adjusted system paths!")


Cloning repository...


Cloning into 'Facial-Expression-Recognition'...


Successfully linked GitHub repository and adjusted system paths!


In [3]:
import wandb

wandb.login(key="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")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: kende23 (kende23-n-a) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
@dataclass
class Config:
    csv_path: str = "/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv"

    batch_size: int = 64
    lr: float = 1e-3
    epochs: int = 20

    num_classes: int = 7
    image_size: int = 48

    random_seed: int = 67


config = Config()

In [5]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(config.random_seed)

In [6]:
sweep_config = {

    "method": "random",

    "metric": {
        "name": "val_acc",
        "goal": "maximize"
    },


    "parameters": {

        "lr": {
            "values": [
                1e-3,
                5e-4,
                1e-4
            ]
        },


        "dropout": {
            "values": [
                0.2,
                0.3,
                0.5
            ]
        },


        "batch_size": {
            "values": [
                32,
                64
            ]
        },


        "epochs": {
            "value":20
        }
    }
}

In [7]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(device)

cuda


In [8]:
df = pd.read_csv(config.csv_path)

print(df.shape)
df.head()

(28709, 2)


,emotion,pixels
0,0,70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...
1,0,151 150 147 155 148 133 111 140 170 174 182 15...
2,2,231 212 156 164 174 138 161 173 182 200 106 38...
3,4,24 32 36 30 32 23 19 20 30 41 21 22 32 34 21 1...
4,6,4 0 0 0 0 0 0 0 0 0 0 0 3 15 23 28 48 50 58 84...


In [9]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=config.random_seed,
    stratify=df["emotion"]
)

print(len(train_df))
print(len(val_df))

22967
5742


In [10]:
class_counts = train_df["emotion"].value_counts().sort_index()

print(class_counts)

emotion
0    3196
1     349
2    3277
3    5772
4    3864
5    2537
6    3972
Name: count, dtype: int64


In [11]:
class_weights = 1.0 / torch.tensor(
    class_counts.values,
    dtype=torch.float32
)

class_weights = class_weights / class_weights.sum()

print(class_weights)
class_weights = class_weights.to(device)

tensor([0.0686, 0.6282, 0.0669, 0.0380, 0.0567, 0.0864, 0.0552])


In [12]:
def pixels_to_image(pixel_string):
    pixels = np.fromstring(
        pixel_string,
        dtype=np.uint8,
        sep=" "
    )

    return pixels.reshape(48, 48)

In [13]:
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

In [14]:
class FERDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
    

    def __len__(self):
        return len(self.dataframe)


    def __getitem__(self, idx):

        row = self.dataframe.iloc[idx]

        image = pixels_to_image(row["pixels"])

        image = image.astype(np.uint8)


        if self.transform:
            image = self.transform(image)

        else:
            image = image.astype(np.float32) / 255.0
            image = torch.tensor(image).unsqueeze(0)


        label = torch.tensor(
            row["emotion"],
            dtype=torch.long
        )

        return image, label

In [15]:
train_dataset = FERDataset(train_df)
val_dataset = FERDataset(val_df)

In [16]:
class ResidualBlock(nn.Module):

    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels,
            out_channels,
            kernel_size=3,
            padding=1,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(out_channels)


        # shortcut connection
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )

        else:
            self.shortcut = nn.Identity()


    def forward(self, x):

        identity = self.shortcut(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = torch.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += identity

        out = torch.relu(out)

        return out

In [17]:
class SmallResNet(nn.Module):

    def __init__(self, dropout=0.3):
        super().__init__()


        self.features = nn.Sequential(

            nn.Conv2d(
                1,
                16,
                kernel_size=3,
                padding=1,
                bias=False
            ),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            ResidualBlock(16,16),
            nn.MaxPool2d(2),
            ResidualBlock(16,32,stride=2),
            ResidualBlock(32,64,stride=2)
        )
        
        self.dropout = nn.Dropout(dropout)
        
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(64,7)
        )


    def forward(self,x):

        x = self.features(x)

        x = self.classifier(x)

        return x

In [18]:
def train_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad() #?

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / len(loader.dataset)

    return epoch_loss, epoch_acc

In [19]:
def validate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / len(loader.dataset)

    return epoch_loss, epoch_acc

In [20]:
def train():

    run = wandb.init()

    cfg = wandb.config

    model = SmallResNet(
        dropout=cfg.dropout
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=cfg.lr
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    criterion = nn.CrossEntropyLoss(
        weight=class_weights
    )

    best_val_acc = 0

    for epoch in range(config.epochs):

        train_loss, train_acc = train_epoch(
            model,
            train_loader,
            criterion,
            optimizer,
            device
        )

        val_loss, val_acc = validate(
            model,
            val_loader,
            criterion,
            device
        )

        wandb.log({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc
        })

        print(
            f"Epoch {epoch+1}/{config.epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

        if val_acc > best_val_acc:

            best_val_acc = val_acc

            torch.save(
                model.state_dict(),
                "best_baseline_model.pt"
            )

    artifact = wandb.Artifact(
        "best-model",
        type="model"
    )

    artifact.add_file(
        "best_baseline_model.pt"
    )

    wandb.log_artifact(artifact)

    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():

        for x, y in val_loader:

            x = x.to(device)

            outputs = model(x)

            preds = outputs.argmax(dim=1)

            y_true.extend(y.numpy())
            y_pred.extend(preds.cpu().numpy())

    report = classification_report(
        y_true,
        y_pred,
        output_dict=True
    )

    wandb.log({
        "precision_macro": report["macro avg"]["precision"],
        "recall_macro": report["macro avg"]["recall"],
        "f1_macro": report["macro avg"]["f1-score"]
    })

    print(
        classification_report(
            y_true,
            y_pred
        )
    )

    run.finish()

In [ ]:
sweep_id = wandb.sweep(
    sweep_config,
    project="fer2013"
)

wandb.agent(
    sweep_id,
    function=train,
    count=20
)

Create sweep with ID: xrqem4k9
Sweep URL: https://wandb.ai/kende23-n-a/fer2013/sweeps/xrqem4k9


wandb: Agent Starting Run: 41rbh2t2 with config:
wandb: 	batch_size: 64
wandb: 	dropout: 0.3
wandb: 	epochs: 20
wandb: 	lr: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Epoch 1/20 | Train Loss: 1.9175 | Train Acc: 0.2061 | Val Loss: 1.8733 | Val Acc: 0.2485
Epoch 2/20 | Train Loss: 1.8594 | Train Acc: 0.2486 | Val Loss: 1.8157 | Val Acc: 0.3114
Epoch 3/20 | Train Loss: 1.7897 | Train Acc: 0.3006 | Val Loss: 1.7546 | Val Acc: 0.3568
Epoch 4/20 | Train Loss: 1.7266 | Train Acc: 0.3325 | Val Loss: 1.7039 | Val Acc: 0.3607
Epoch 5/20 | Train Loss: 1.6683 | Train Acc: 0.3546 | Val Loss: 1.7256 | Val Acc: 0.3875
Epoch 6/20 | Train Loss: 1.6048 | Train Acc: 0.3723 | Val Loss: 1.6220 | Val Acc: 0.3619
Epoch 7/20 | Train Loss: 1.5501 | Train Acc: 0.3925 | Val Loss: 1.6006 | Val Acc: 0.4187
Epoch 8/20 | Train Loss: 1.4933 | Train Acc: 0.4064 | Val Loss: 1.5728 | Val Acc: 0.4023
Epoch 9/20 | Train Loss: 1.4467 | Train Acc: 0.4199 | Val Loss: 1.5642 | Val Acc: 0.4316
Epoch 10/20 | Train Loss: 1.3907 | Train Acc: 0.4337 | Val Loss: 1.5494 | Val Acc: 0.4124
Epoch 11/20 | Train Loss: 1.3520 | Train Acc: 0.4460 | Val Loss: 1.5118 | Val Acc: 0.4211
Epoch 12/20 | Train

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
f1_macro,▁
precision_macro,▁
recall_macro,▁
train_acc,▁▂▃▄▄▅▅▅▆▆▆▇▇▇▇▇████
train_loss,█▇▇▆▆▅▅▄▄▃▃▃▂▂▂▂▂▁▁▁
val_acc,▁▃▅▅▆▅▇▆▇▆▇▇▇█▇▇████
val_loss,█▇▆▅▅▃▃▂▂▂▁▂▁▂▁▁▁▁▂▂
epoch,20
f1_macro,0.42636
precision_macro,0.422


wandb: Agent Starting Run: jt4zpeyt with config:
wandb: 	batch_size: 32
wandb: 	dropout: 0.3
wandb: 	epochs: 20
wandb: 	lr: 0.001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


...

Epoch 1/20 | Train Loss: 1.9060 | Train Acc: 0.2091 | Val Loss: 1.8570 | Val Acc: 0.2400
Epoch 2/20 | Train Loss: 1.8314 | Train Acc: 0.2731 | Val Loss: 1.8053 | Val Acc: 0.2955
Epoch 3/20 | Train Loss: 1.7408 | Train Acc: 0.3308 | Val Loss: 1.7204 | Val Acc: 0.3255
Epoch 4/20 | Train Loss: 1.6620 | Train Acc: 0.3650 | Val Loss: 1.6653 | Val Acc: 0.3487
Epoch 5/20 | Train Loss: 1.6008 | Train Acc: 0.3878 | Val Loss: 1.5916 | Val Acc: 0.4117
Epoch 6/20 | Train Loss: 1.5322 | Train Acc: 0.4093 | Val Loss: 1.5627 | Val Acc: 0.3990
Epoch 7/20 | Train Loss: 1.4767 | Train Acc: 0.4284 | Val Loss: 1.5148 | Val Acc: 0.4073
Epoch 8/20 | Train Loss: 1.4184 | Train Acc: 0.4422 | Val Loss: 1.4980 | Val Acc: 0.4282
Epoch 9/20 | Train Loss: 1.3635 | Train Acc: 0.4645 | Val Loss: 1.4829 | Val Acc: 0.4363
Epoch 10/20 | Train Loss: 1.3244 | Train Acc: 0.4776 | Val Loss: 1.4706 | Val Acc: 0.4171
Epoch 11/20 | Train Loss: 1.2841 | Train Acc: 0.4874 | Val Loss: 1.4559 | Val Acc: 0.4363
Epoch 12/20 | Train

wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: 3lng12hl with config:
wandb: 	batch_size: 64
wandb: 	dropout: 0.3
wandb: 	epochs: 20
wandb: 	lr: 0.0005
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Epoch 1/20 | Train Loss: 1.8816 | Train Acc: 0.2264 | Val Loss: 1.8012 | Val Acc: 0.3323
Epoch 2/20 | Train Loss: 1.7036 | Train Acc: 0.3470 | Val Loss: 1.6511 | Val Acc: 0.3669
Epoch 3/20 | Train Loss: 1.5628 | Train Acc: 0.4044 | Val Loss: 1.6162 | Val Acc: 0.3572
Epoch 4/20 | Train Loss: 1.4582 | Train Acc: 0.4337 | Val Loss: 1.4806 | Val Acc: 0.4368
Epoch 5/20 | Train Loss: 1.3656 | Train Acc: 0.4664 | Val Loss: 1.4584 | Val Acc: 0.4291
Epoch 6/20 | Train Loss: 1.2824 | Train Acc: 0.4872 | Val Loss: 1.4881 | Val Acc: 0.4040
Epoch 7/20 | Train Loss: 1.2185 | Train Acc: 0.5111 | Val Loss: 1.4658 | Val Acc: 0.4897
Epoch 8/20 | Train Loss: 1.1615 | Train Acc: 0.5231 | Val Loss: 1.4455 | Val Acc: 0.4821
Epoch 9/20 | Train Loss: 1.1021 | Train Acc: 0.5480 | Val Loss: 1.4390 | Val Acc: 0.4939
Epoch 10/20 | Train Loss: 1.0709 | Train Acc: 0.5568 | Val Loss: 1.5321 | Val Acc: 0.5141
Epoch 11/20 | Train Loss: 1.0347 | Train Acc: 0.5709 | Val Loss: 1.6296 | Val Acc: 0.5169
Epoch 12/20 | Train

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
f1_macro,▁
precision_macro,▁
recall_macro,▁
train_acc,▁▃▄▄▅▅▅▆▆▆▆▇▇▇▇▇████
train_loss,█▇▆▅▅▄▄▃▃▃▃▂▂▂▂▂▁▁▁▁
val_acc,▁▂▂▅▅▄▇▇▇██▆▆▇▆▆█▇▇█
val_loss,█▅▄▂▁▂▂▁▁▃▅▂▇▄▄▆▆▆█▇
epoch,20
f1_macro,0.48762
precision_macro,0.51917


wandb: Sweep Agent: Waiting for job.


In [ ]:
wandb.finish()